# Demo 3 · Pinned versus latest

**Slides:** "“latest” is a moving target" (slide 9), "Pin it, log it, cache it" (slide 15), demo step 03.

The same small evaluation runs against a pinned Gemini model and against the `-latest` alias.
Every response records which model version answered, and responses are cached so a rerun makes no API calls.

Needs a Gemini API key stored as a Colab secret named `GOOGLE_API_KEY` (key icon in the left sidebar).
Runtime: any. Takes under a minute.

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
!pip install -q -U google-genai

## 1 · Connect and choose a pinned model

List the available Flash models and copy a **stable, explicitly versioned** ID into `PINNED_MODEL`.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
for m in client.models.list():
    if "flash" in m.name:
        print(m.name, "|", m.version)

In [ ]:
PINNED_MODEL = "REPLACE-with-a-stable-model-id"   # e.g. a versioned Flash model from the list above
LATEST_ALIAS = "gemini-flash-latest"
assert not PINNED_MODEL.startswith("REPLACE"), "Set PINNED_MODEL to a model ID from the list above"

CONFIG = types.GenerateContentConfig(temperature=0, seed=1234)

## 2 · Ask with logging and a cache

The cache key is a hash of model, prompt and config. Changing any of them is a new request; repeating them is free and identical.

In [ ]:
import hashlib, json, pathlib

CACHE = pathlib.Path(".gemini_cache/responses.jsonl")
CACHE.parent.mkdir(exist_ok=True)
cache = {}
if CACHE.exists():
    for line in CACHE.read_text().splitlines():
        rec = json.loads(line)
        cache[rec["key"]] = rec

def ask(model: str, prompt: str) -> dict:
    config = CONFIG.model_dump(exclude_none=True)
    key = hashlib.sha256(json.dumps([model, prompt, config], sort_keys=True).encode()).hexdigest()
    if key in cache:
        return {**cache[key], "cached": True}
    response = client.models.generate_content(model=model, contents=prompt, config=CONFIG)
    rec = {"key": key, "model": model, "model_version": response.model_version,
           "prompt": prompt, "config": config, "text": (response.text or "").strip()}
    cache[key] = rec
    with CACHE.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    return {**rec, "cached": False}

## 3 · A tiny evaluation set

In [ ]:
EVAL = [
    ("The update fixed every crash I had reported.", "positive"),
    ("The battery now lasts half as long as before.", "negative"),
    ("The package arrived on Tuesday.", "neutral"),
    ("Support answered quickly but could not solve the problem.", "negative"),
    ("Setup took two minutes and everything just worked.", "positive"),
    ("The manual is available in three languages.", "neutral"),
    ("I would not buy this again.", "negative"),
    ("Honestly better than I expected for the price.", "positive"),
]
TEMPLATE = ("Classify the sentiment of this review. Answer with exactly one word: "
            "positive, negative or neutral.\n\nReview: {text}")

## 4 · Run against the pinned model and the alias

In [ ]:
import pandas as pd

def evaluate(model):
    rows = []
    for text, expected in EVAL:
        r = ask(model, TEMPLATE.format(text=text))
        answer = r["text"].lower().strip(" .")
        rows.append({"review": text[:40], "expected": expected, "answer": answer,
                     "correct": answer == expected, "model_version": r["model_version"],
                     "cached": r["cached"]})
    return pd.DataFrame(rows)

pinned = evaluate(PINNED_MODEL)
latest = evaluate(LATEST_ALIAS)
print(f"pinned  {PINNED_MODEL}: accuracy {pinned.correct.mean():.2f}, served by {set(pinned.model_version)}")
print(f"latest  {LATEST_ALIAS}: accuracy {latest.correct.mean():.2f}, served by {set(latest.model_version)}")
pd.concat({"pinned": pinned, "latest": latest}, axis=1)

## 5 · Rerun: everything comes from the cache

Same answers, zero API calls. This is what makes an LLM evaluation reproducible next month.

In [ ]:
again = evaluate(PINNED_MODEL)
print("all cached:", again.cached.all(), "| identical answers:", (again.answer == pinned.answer).all())

## 6 · Log the evaluation

In [ ]:
!pip install -q mlflow
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("devfest-gemini-eval")
for label, model, df in [("pinned", PINNED_MODEL, pinned), ("latest", LATEST_ALIAS, latest)]:
    with mlflow.start_run(run_name=label):
        mlflow.log_params({"model": model, "model_version": ",".join(sorted(set(df.model_version))),
                           "temperature": 0, "seed": 1234, "n_items": len(df)})
        mlflow.log_metric("accuracy", float(df.correct.mean()))
        mlflow.log_artifact(str(CACHE))
print("logged")

## What to say

The alias and the pinned ID may give the same answers today. The point is the record: when the alias moves,
the logged `model_version` shows it, and the cache lets you rerun last month's evaluation exactly.
Upgrade the pinned ID deliberately, in a pull request, with this evaluation rerun.